In [1]:
import pandas as pd #importamos a principal biblioteca que utilizaremos
import numpy as np
import matplotlib.pyplot as plt
from datetime import date
from dateutil.relativedelta import relativedelta

In [90]:
def cria_taxa_inadimplencia(data,faixa_etaria):
    n_elementos_totais = int(data.shape[0])
    elementos_presentes = data["status"].value_counts()
    n_inadimplentes = int(elementos_presentes["inadimplente"])
    return {"Faixa_etaria":faixa_etaria, "Total_clientes":n_elementos_totais, "Inadimplentes":n_inadimplentes, "Taxa de Inadimplência":(n_inadimplentes/n_elementos_totais)}

In [217]:
def informacao_emprestimo(data,cliente):
    separa_per_emprestimo = {separa_per_emprestimo: grupo for separa_per_emprestimo, grupo in data.groupby('id_emprestimo')}
    lista_emprestimos = list(separa_per_emprestimo.keys())
    informacao_cliente = {"id_cliente":cliente,"Valor total pego":0,"Valor total pago":0,"Valor restante":0,"Percentual Quitado":0}
    for i in range(len(lista_emprestimos)):
        valor = separa_per_emprestimo[lista_emprestimos[i]]["valor"].unique()
        valor_pego = int(valor[0])
        informacao_cliente["Valor total pego"] += valor_pego
        valor_pago = int(separa_per_emprestimo[lista_emprestimos[i]]["valor_pago"].sum())
        informacao_cliente["Valor total pago"] += valor_pago
        valor_restante = valor_pego - valor_pago
        informacao_cliente["Valor restante"] += valor_restante
    informacao_cliente["Percentual Quitado"] += informacao_cliente["Valor total pago"]/informacao_cliente["Valor total pego"]
    return informacao_cliente

In [91]:
#Importamos os dados estudados
clientes = pd.read_csv("Clientes.csv")
emprestimos = pd.read_csv("Emprestimos.csv")
pagamentos = pd.read_csv("Pagamentos.csv")

In [139]:
#clientes.head()

,id_cliente,nome,data_nascimento,cidade,data_cadastro
0,1,Cliente_1,1959-12-28,Fortaleza,2018-04-13
1,2,Cliente_2,1974-09-04,Rio de Janeiro,2020-09-30
2,3,Cliente_3,1962-07-09,Fortaleza,2019-02-24
3,4,Cliente_4,1998-12-05,Salvador,2018-12-23
4,5,Cliente_5,1987-11-08,São Paulo,2018-05-11


In [140]:
emprestimos.head()

,id_emprestimo,id_cliente,valor,taxa_juros,data_contratacao,status
0,1,411,40277,0.037,2021-05-31,quitado
1,2,2765,45236,0.022,2020-02-18,inadimplente
2,3,2190,12713,0.022,2022-05-04,ativo
3,4,3660,22275,0.022,2020-03-13,inadimplente
4,5,915,24412,0.039,2022-12-24,inadimplente


In [94]:
#pagamentos.head()

In [143]:
#criamos um dataframe com a data de hoje
date_atual = pd.DataFrame(data = {"id_cliente":np.arange(0,5000),"data_atual": ["2026-02-26"]*5000})
date_atual["data_atual"] = pd.to_datetime(date_atual["data_atual"])
#criamos um data com as idades
anos = (((date_atual["data_atual"]-pd.to_datetime(clientes["data_nascimento"]))/pd.Timedelta(days=1)).astype(int))//365
#Vamos criar dataframes com as informações importantes de cada um dos dataframes
data0 = pd.DataFrame(data = {"id_cliente":clientes["id_cliente"], "Idade":anos, "cidade":clientes["cidade"]})
data1 = pd.DataFrame(data = {"id_emprestimo":emprestimos["id_emprestimo"],"id_cliente":emprestimos["id_cliente"],"valor":emprestimos["valor"].astype(int), "status":emprestimos["status"]})
data2 = pd.DataFrame(data = {"id_emprestimo":pagamentos["id_emprestimo"], "valor_pago":pagamentos["valor_pago"]})
#vamos dar um merge para juntar as colunas que são interessantes de ter
data3 = pd.merge(data0,data1, how = "inner", on = "id_cliente")
data4 = pd.merge(data3,data2, how = "inner", on = "id_emprestimo")
#Banco de dados que iremos utilizar para resolver os problemas
data_opera = data4

In [144]:
data_opera

,id_cliente,Idade,cidade,id_emprestimo,valor,status,valor_pago
0,1,66,Fortaleza,1487,47550,ativo,5336
1,3,63,Fortaleza,4874,2698,quitado,7453
2,3,63,Fortaleza,4874,2698,quitado,6970
3,3,63,Fortaleza,4874,2698,quitado,8490
4,5,38,São Paulo,102,17498,quitado,2501
...,...,...,...,...,...,...,...
4995,4996,23,Rio de Janeiro,2029,33223,quitado,8439
4996,4996,23,Rio de Janeiro,2029,33223,quitado,9120
4997,4996,23,Rio de Janeiro,2029,33223,quitado,5667
4998,4996,23,Rio de Janeiro,2029,33223,quitado,1281


In [145]:
#Calculo do Valor emprestado por cidade.
cidades = {cidades: grupo for cidades, grupo in data_opera.groupby('cidade')}
chaves = list(cidades.keys())
total_emprestado = []
for i in range(len(chaves)):
    valor = int(cidades[chaves[i]]['valor'].sum())
    linha = {"cidade":chaves[i], "total_emprestado":valor}
    total_emprestado.append(linha)
valor_emprestado = pd.DataFrame(data=total_emprestado) 

In [146]:
valor_emprestado

,cidade,total_emprestado
0,Belo Horizonte,16880090
1,Brasília,17634993
2,Curitiba,20223714
3,Fortaleza,18717681
4,Rio de Janeiro,17780392
5,Salvador,18556349
6,São Paulo,18365137


In [147]:
#taxa de inadimplencia por faixa etaria
#18-25
dado_intervalo1 = data_opera[(data_opera["Idade"]<25) & (data_opera["Idade"]>=18)]
resultado1 = cria_taxa_inadimplencia(dado_intervalo1,"18-25")
#25-40
dado_intervalo2 = data_opera[(data_opera["Idade"]<=40) & (data_opera["Idade"]>=25)]
resultado2 = cria_taxa_inadimplencia(dado_intervalo2,"25-40")
#41-60
dado_intervalo3 = data_opera[(data_opera["Idade"]<60) & (data_opera["Idade"]>=41)]
resultado3 = cria_taxa_inadimplencia(dado_intervalo3,"41-60")
#60+
dado_intervalo4 = data_opera[data_opera["Idade"]>=60]
resultado4 = cria_taxa_inadimplencia(dado_intervalo4,"60+")
#Criação do Dataframe.
inadimplecia_taxa = pd.DataFrame(data = [resultado1, resultado2, resultado3, resultado4])

In [148]:
inadimplecia_taxa

,Faixa_etaria,Total_clientes,Inadimplentes,Taxa de Inadimplência
0,18-25,343,114,0.332362
1,25-40,1552,505,0.325387
2,41-60,1668,566,0.339329
3,60+,1437,460,0.320111


In [220]:
#Para os emprestimos ativos.
separa_per_status = {separa_per_status: grupo for separa_per_status, grupo in data_opera.groupby('status')}
dados_ativos = separa_per_status["ativo"]
#separando os clientes ativos.
separa_per_clientes = {separa_per_clientes: grupo for separa_per_clientes, grupo in dados_ativos.groupby('id_cliente')}
chaves = list(separa_per_clientes.keys())
#calculando a informação de cada cliente
data_emprestimo = []
for i in range(len(chaves)):
    dados_cliente_ativo = informacao_emprestimo(separa_per_clientes[chaves[i]],chaves[i])
    data_emprestimo.append(dados_cliente_ativo)
data_emprestimos_ativos = pd.DataFrame(data = data_emprestimo)

In [221]:
data_emprestimos_ativos

,id_cliente,Valor total pego,Valor total pago,Valor restante,Percentual Quitado
0,1,47550,5336,42214,0.112219
1,21,26162,8629,17533,0.329830
2,23,24809,6650,18159,0.268048
3,28,34826,4749,30077,0.136364
4,30,21297,4323,16974,0.202986
...,...,...,...,...,...
947,4975,35766,2562,33204,0.071632
948,4980,16878,9568,7310,0.566892
949,4985,30612,1399,29213,0.045701
950,4989,38437,301,38136,0.007831
